# Chapter 12 &mdash; A PDA for $L_{Dyck}$, and its Simulation

**Concept 3 of the Chapter 12 decomposition:** *A PDA for $L_{Dyck}$, and its Simulation*

Push on `(`, pop on `)`, accept when the input is gone and `#` is on top.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-PDA-For-Dyck/Concept-PDA-For-Dyck.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The design, in three lines of reasoning:

* an unmatched `(` is a **debt** &mdash; push a marker;
* a `)` **pays** a debt &mdash; pop a marker; if there is nothing to pop the run dies,
  which is exactly the **prefix condition**;
* at the end the stack must hold **only** `#` &mdash; that is the **count condition**.

The two Dyck conditions of Chapter 11 map one-to-one onto two features of the machine.
That correspondence is the point of the example.

Simulate it and watch the stack height trace the same hill/valley curve as the plot in
Chapter 11, Concept 8.

## 2. Definitions

### The machine

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
Dyck = md2mc('''PDA
!! Push on '(', pop on ')', accept when the input is gone and # is on top.
I : ( , #  ; (#  -> I     !! first '(' -- push it above the bottom marker
I : ( , (  ; ((  -> I     !! another '(' -- push
I : ) , (  ; ''  -> I     !! ')' matches -- pop
I : '' , # ; #   -> F     !! nothing left and stack is just # -- accept
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### A hand simulation, tracking the stack

In [ ]:
def simulate(s):
    st = ['#']
    print("  start          stack %s" % ''.join(st))
    for ch in s:
        if ch == '(':
            st.append('(')
        else:
            if st[-1] != '(':
                print("  read ')'       stack %s  <- nothing to pop: DIE" % ''.join(st))
                return False
            st.pop()
        print("  read '%s'       stack %s" % (ch, ''.join(st)))
    ok = st == ['#']
    print("  end            stack %s  -> %s" % (''.join(st), "ACCEPT" if ok else "reject"))
    return ok

## 3. Tests

A successful run.

In [ ]:
ok = simulate('(())')
assert ok == pda_accepts(Dyck, '(())', STKMAX=8)

The **prefix condition** is the pop that finds nothing.

In [ ]:
ok = simulate('())')
assert not ok and not pda_accepts(Dyck, '())', STKMAX=8)

The **count condition** is the stack not being clean at the end.

In [ ]:
ok = simulate('(()')
assert not ok and not pda_accepts(Dyck, '(()', STKMAX=8)

Stack height traces the same curve as Chapter 11's hill/valley plot.

In [ ]:
def heights(s):
    h, out = 0, [0]
    for ch in s:
        h += 1 if ch == '(' else -1
        out.append(h)
    return out
for s in ['(())', '()()', '(()())']:
    print("  %-9s heights %s" % (s, heights(s)))
print("\nnever negative, ends at 0 -- exactly the two Dyck conditions.")

Agreement with the specification, exhaustively.

In [ ]:
from itertools import product
def balanced(s):
    d = 0
    for ch in s:
        d += 1 if ch == '(' else -1
        if d < 0: return False
    return d == 0
strs = [''.join(p) for k in range(7) for p in product('()', repeat=k)]
bad = [s for s in strs if pda_accepts(Dyck, s, STKMAX=9) != balanced(s)]
print("mismatches over %d strings :" % len(strs), bad)
assert not bad

## 4. Animation

The Dyck PDA. Step through `(()())` and watch the debts accumulate and clear.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(Dyck, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Modify it for two bracket kinds. How many stack symbols do you need?
2. Which single line enforces the prefix condition?
3. Rewrite it to accept by **empty stack** rather than final state.

In [ ]:
# Your work for the exercises above.